# RAG (Retrieval Augmented Generation)

Ok so this is what I was looking forward to. It is by far the most interesting part about this etude for me since it's not a complete black box, like most of prompting was.

I already now the basic concepts behind RAG, but I want to really dive deep into this. I'll break down the concepts and try and implement the math from scratch, from the formulas to a full implementation. I might now fully accomplish that, but I'll do it till I think I really understand the concepts behind it.

---

So RAG is a technique to handle getting data from an external source, like a document, that is too big for a prompt.

If you have a huge doc, like 1000 pages, you shouldn't really just drop all the text in the prompt. Problems with that are:
- **Attention is quadratic:** this basically means that for every word the model has to process, it has to go over each other word to understand how much it should "attend" to it. This means that for each of the $n$ tokens, we need to go over $n$ tokens, which gives us a complexity of $O(n^2)$.

- **Lost in the Middle:** while self-attention on large context is more of a mechanical problem that can be brute forced with more computing power, "lost in the middle" is a problem well documented in the paper with same name ([Lost in the Middle](https://arxiv.org/abs/2307.03172)). The problem here is that attention get's distributed unevenly across the context, so the model has a hard time handling information that is at the middle of the context (like page 500 of 1000 pages) and privileges information on the extremes (like page 10 or 990).

- **Cost and Latency:** as we've seen previously, models are stateless. That means that every new prompt you send out should include the whole 1000 pages over.

This is just to name a few.

What RAG means in practice is, based on the prompt, gather the relevant parts from documents and then use those specific relevant parts on the prompt.
In a real world scenario, if my prompt asks for something that is in chapter 10 of a book (let's say page 760 to 762 of our 1000 pages document), through some strategies I'll cover later, the workflow will first *retrieve* the relevant pages, add it as context and finally send it to the model to process.

RAG is pretty awesome, but there are somethings we have to weight when thinking about using it:
- it requires preprocessing step to chunk documents;
- needs a search mechanism to find those documents (in practice this means having a vector DB setup).

Generally, RAG is the go to when dealing with large or multiple documents.

## Chunking Strategies

So in order to handle a huge document, we need to first chunk it. Here I'll cover some of the most common chunking strategies.

### Size-Based Chunking

This is exactly what is sounds like. Let's say we define our chunk should have 200 words. Then if we have a document with 1000 words we'll have 5 chunks. The problem is that chunks might loose context from text that's was left behind on a previous chunk.

| Chunks   | Content |
|   :---   |  ---:  |
| Chunk #1 | "To execute procedure X we " |
| Chunk #2 | "need to make sure we have " |
| Chunk #3 | "covered all requires steps." |

You can see how this is a problem (even with my oversimplified example).

A solution to this is add an extra margin for each chunk.

| Chunks   | Content |
|   :---   |  ---:  |
| Chunk #1 | "To execute procedure X we need to make sure we have " |
| Chunk #2 | "To execute procedure X we need to make sure we have covered all requires steps." |
| Chunk #3 | "need to make sure we have covered all requires steps." |

### Structure-Based Chunking

This is for me by far the clearer one to exemplify and the first one that comes to my mind when I think about chunking a document.

It's basically chunking using the documents existing structure. So like, headers, sections, paragraphs.

Some problems with structure-based chunking:
- Some documents might completely lack structural markers
- Structure is usually unevenly distributed, so one section might be 5 paragraphs and another might be 10 pages

### Semantic-Based Chunking

This is the most demanding but possibily the best in terms of accuracy. For it, we divide text into sentences and use NLP to determine how related a sentence is to the previous one. That way we can have chunks with only sentences that are relevant between themselves.

### Sentence-Based Chunking

This is similar to the size-based where we have the overlap, but of a fixed size we chunk based on the sententes.

Example, each chunk allows for 10 sententes and we keep an overlap of 2 sentences between chunks:

| First Sentence # | Last Sentence # |
| ---:             |            :--- |
| 1 | 10 |
| 8 | 18 |
| 16 | 26 | 

## Embedding

Embedding is the process of transforming a text into a vector. That vector is not random, it actually holds semantic value. We have models that already does embedding for us and that's what I'll be using.

Like I mentioned, embedding holds the semantic value of text, and that's why we use these vector values for semantic search.

Let's say we have these chunks and their respective embeddings:

| Chunk | Embedding |
|---|---|
| Chunk #1 | [0.12, -0.45, 0.33, 0.78, -0.09, 0.61, -0.27, 0.15, -0.52, 0.94] |
| Chunk #2 | [0.03, 0.87, -0.14, -0.62, 0.41, 0.29, -0.75, 0.08, 0.56, -0.31] |
| Chunk #3 | [-0.68, 0.22, 0.49, -0.11, 0.73, -0.38, 0.05, 0.91, -0.47, 0.16] |
| Chunk #4 | [0.34, -0.59, 0.02, 0.66, -0.23, 0.88, -0.41, 0.19, -0.07, 0.53] |
| Chunk #5 | [-0.21, 0.44, -0.76, 0.13, 0.60, -0.05, 0.37, -0.82, 0.29, 0.09] |

Now let's say we processed the prompt and got this embedding:
[-0.71, 0.19, 0.52, -0.14, 0.70, -0.35, 0.08, 0.88, -0.44, 0.20]

If we were to plot these vector we could see something like this:

<img src="../assets/similarity_vectors.png" width="700px">

Looking at it like that we can see that Chunk #3 seems to be the most semantically relevant to our prompt, given the similarity between them. This is of course just a visualization. The way we actually calculate it is with Cosine Similarity.

### Cosine Similarity

Cosine similarity determines how similar two vectors are. The math formula for it is $$\cos(\theta) = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

Even tho it looks scary like that it's actually pretty simple.

- $A$ and $B$ are the vectors we want to compare.
- $n$ is the dimension of the vector (how many number in the vector).

The numerator (the upper portion of the division) basically tells us to multiply each value of $A$ with the value in the same position on $B$, them sum it all.

In [1]:
chunk_3 = [-0.68, 0.22, 0.49, -0.11, 0.73, -0.38, 0.05, 0.91, -0.47, 0.16]
prompt = [-0.71, 0.19, 0.52, -0.14, 0.70, -0.35, 0.08, 0.88, -0.44, 0.20]

In [2]:
numerator = 0
for i in range(len(chunk_3)):
    numerator += chunk_3[i] * prompt[i]

numerator

2.4824

And for the denominator we multiply the squareroot of the sum of each item in $A$ squared, by the same operation on $B$.

In [3]:
from math import sqrt

first_root = 0
for item in chunk_3:
    first_root += item**2
first_root = sqrt(first_root)

second_root = 0
for item in prompt:
    second_root += item**2
second_root = sqrt(second_root)

denominator = first_root*second_root

denominator

2.487067256830824

Finally we just divide the numerator by the denominator.

In [ ]:
cosine = numerator / denominator
cosine

0.9981233893783914

This number represents how close Chunk #3 is from the Prompt. Very close indeed.

Now, the code I wrote above is abominable. It work sure but there are easier ways to do that using numpy.

In [ ]:
import numpy as np

chunk_3 = np.array(chunk_3)
prompt = np.array(prompt)

numerator = np.sum(chunk_3 * prompt)
numerator

np.float64(2.4823999999999997)

So far so good. The `*` performes an element-wise multiplication on the vector, which is exactly what we achived before, but now way more optimized.
Same for the `.sum()`, which sums all the elements in the resulting vector.

In [ ]:
denominator = np.sqrt(np.sum(chunk_3**2)) * np.sqrt(np.sum(prompt**2))
denominator

np.float64(2.487067256830824)

Here I used `**` which does element-wise exponentiation.

In [ ]:
cosine = numerator / denominator
cosine

np.float64(0.9981233893783912)

Same result, but much cleaner code.

Now there's something that took me while to figure out, but the numerator summation is basically matrix multiplication. $$A \cdot B$$ With matrix multiplication, we multiply each value on matrix A **row** by the corresponding value on the second matrix **column**, then sum them. This is exactly what we were doing with the previous formula. The matrix multiplication operation is also know as dot product. In numpy we can use either `np.dot()` or simply the `@` operator.

In [10]:
numerator = chunk_3 @ prompt
numerator

np.float64(2.4823999999999997)

For the denominator, we can also simplify on how we think of it. What we are doing is squaring all the numbers and summing them up. This is literally the Pythagorean Theorem:

$$a^2 + b^2 = c^2$$

Of course this is just for two dimensions. Our embedding has more dimensions. We can think of it like this:

$$d^2 = x_1^2 + x_2^2 + x_3^2 + x_4^2 + ... + x_n^2$$

The above expression can be simplified as:

$$d^2 = \sum_{i=1}^{n} x_i^2$$

Now we can see that $d$ is squared. This is because Pythagorean Theorem actually leaves the answer squared. However the cosine similarity equation we saw before actually takes the square root of the value. In math this is actually it's own equation known as Eucledian Norm. While Pythagorean Theorem is $d^2 = \sum_{i=1}^{n} x_i^2$, Euclidian Norm is just:

$$\Vert x \Vert = \sqrt{\sum_{i=1}^{n} x_i^2}$$

As expected, numpy does provide a `.norm()` operation, which makes our code even simpler.

In [11]:
denominator = np.linalg.norm(chunk_3) * np.linalg.norm(prompt)
denominator

np.float64(2.487067256830824)

The end result is this:

$$\frac{A \cdot B}{\Vert A \Vert \Vert B \Vert}$$

Much more elegant.

In [13]:
cosine = chunk_3 @ prompt / (np.linalg.norm(chunk_3) * np.linalg.norm(prompt))
cosine

np.float64(0.9981233893783912)

Numpy stops here, however other libs like Scikit Learn, Pytorch and SciPy have builtin Cosine Similarity function.

Now, retriving the document with closest similarity, isn't that straight forward.

In [15]:
chunks = np.array([
    [0.12, -0.45, 0.33, 0.78, -0.09, 0.61, -0.27, 0.15, -0.52, 0.94],
    [0.03, 0.87, -0.14, -0.62, 0.41, 0.29, -0.75, 0.08, 0.56, -0.31],
    [-0.68, 0.22, 0.49, -0.11, 0.73, -0.38, 0.05, 0.91, -0.47, 0.16],
    [0.34, -0.59, 0.02, 0.66, -0.23, 0.88, -0.41, 0.19, -0.07, 0.53],
    [-0.21, 0.44, -0.76, 0.13, 0.60, -0.05, 0.37, -0.82, 0.29, 0.09],
])

prompt = np.array([-0.71, 0.19, 0.52, -0.14, 0.70, -0.35, 0.08, 0.88, -0.44, 0.20])

def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

In [17]:
max = -1
index = -1

for i, chunk in enumerate(chunks):
    cos = cosine(chunk, prompt)
    if cos > max:
        max = cos
        index = i

print(max)
print(index)

0.9981233893783912
2


The above works, but there's an easier way to do it. Going back to matrix multiplication again, each row gets multiplied by the corresponding value on the column, for each row and each column. This means that the dot product handles the entire first part of the equation.

```python
for chunk in chunks:
    chunk @ prompt
```

is the same as

```python
prompt @ chunks.T
```

One small detail here. Like I said, row get's multiplied by column. Our chunks embeddings are as rows, so we need to transpose them to be columns. That's what `.T` does. It doesnt actually move memory around, it just tells numpy to treat rows as columns. So zero computation power to transpose the matrix.

For the numerator, the same applies. When we calculate the norm, it will actually calculate it for each of the rows or columns opn the matrix, depending on the axis we provide. Then when we multiply the prompt norm by the chunk norms, it will do it element-wise. Same for the division at the end.

In [21]:
numerator = prompt @ chunks.T
denominator = np.linalg.norm(prompt) * np.linalg.norm(chunks.T, axis=0)
result = numerator / denominator
index = np.argmax(result)
max = np.max(result)

print(max)
print(index)

0.9981233893783914
2


Ok, this is much better. However, this is not over. While this operation does happen, it's only done for chunks that already somewhere close to the prompt. This is because instead of running the similarity calculation across an entiry vector database, the closest chunks are narrowed down by using an index.

### HNSW

For this section I'll be using [How HNSW Works (Explained from the Original Research Paper)](https://www.youtube.com/watch?v=fV-Stapz0G8) video, by Abid Saudagar.

So the problem at hand is this: we already know how to calculate the distance between one vector and another, so know we just need to find the k nearest chunks to our prompt. This is a problem know as k-NN (k-Nearest Neighbors). What we did before works perfectly. We calculate the distance from to prompt to every other chunk, sort them and we are able to find which are the k nearest chunks. This is know as an exact NN search. The problem is that is scales linearly ($O(n)$). So for 10K vectors, we run 10K operations. For 100K vectors, 100K operations. And so on.


HSNW (Hierarchical Navigatable Small World) is an **approximate** NN search. It does NOT promise the best k nearest matches, but tries to produce the best possible result at a much faster speed. This is because while linear search scales linearly, HNSW scales logarithmically ($O(\log{n})$). The reason why it doesn't ensure the best result is that it's a greedy algorithm, which means it look for the best next move, without planing ahead. Because of this, while looking for the closest next vector, it might find a "local minimum".

<img src="../assets/local_minima.png" width="700px" />

This tradeoff is calculated by the recall value.

$$\text{Recall} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}$$

All that said, how does HNSW work?

The structure HNSW uses is basically a graph, where each node is assigned a max layer. Layer 0 has all the vector, and as we move up they the layers they get progressively smaller, i.e. there as less nodes. Something to note is the nodes are not assign a LAYER, their assigned a MAX layer. This means that a vector with max layer 1 lives in both layer 1 and 0. If a vector with max layer 2 lives in layer 2, 1 and 0.

<img src="../assets/hnsw.png" width="600px"/>

Source of the image (https://github.com/vearch/vearch/wiki/Hnsw-Real-time-Index-Detailed-Design)

In practice, nodes have an ID, a max layer, the vector itself (obviously) and an array of arrays (or better yet, an array of adjencency lists). If a node belongs in more than one layer, it should have one adjencency list for each of its layers.

We also have to control de state of the index. If we think of it as a class, it would have to hold all the nodes, the entry point (this is the node where the search starts), the max layer and some other very important parameters I'll discuss next.

The way searching for a vector works is simple. We start of at the entry point node and calculate its and its neighbors distance to the vector we are querying. If its distance to the vector is less then we stop. Else, we select the neighbor that is closer to the query and repeat the process. This way we greedily find the closest node on that layer. Then we move down a layer, so now instead of checking the neighbors from that node on the $n$ layer, we check the neighbors on the $n-1$ layer. So the further we move down, the closer we get to the query. This is why the node distribution in each layer is different (as discusses previously and as seen in the image above). We repeat this process till we get to layer 0, where all nodes live.

Here the algorithm changes.